# MRO Parts Catalog — Cleaning Pipeline

**Input:** `data/raw_data.csv` (292,658 rows × 87 columns)  
**Output:** `data/cleaned_data.csv` (292,658 rows × 19 columns)

Every decision in this notebook was made after verifying against the full dataset.  
See `eda.ipynb` for the investigation behind each decision.

---
## 0. Imports & Paths

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 80)

RAW_PATH   = 'data/raw_data.csv'
CLEAN_PATH = 'data/cleaned_data.csv'

print('Imports OK')

---
## 1. Load & Rename Columns

Raw column names are a mix of camelCase, dot-notation, and lowercase.
We lowercase everything first, then rename only the columns we are keeping.
Everything else is implicitly dropped by selecting only the renamed columns.

**Notable:** the `price` column contains supplier/distributor names, not prices.
This is a mislabeling in the source export — renamed to `supplier_name`.

In [ ]:
# Maps raw column name -> clean name
# Only columns listed here survive. All others are dropped.
COLUMN_MAP = {
    'item key':                          'supplier_catalog_key',
    'price':                             'supplier_name',
    'sku':                               'sku',
    'attributefamily.code':              'attribute_family',
    'category.default.title':            'category_name',
    'category.id':                       'category_id',
    'brand.default.title':               'brand_name',
    'manufacturer_part_number':          'manufacturer_part_number',
    'last_sold_price':                   'last_sold_price',
    'itemweight':                        'item_weight',
    'attribute_table':                   'attribute_table',
    'downloads':                         'downloads',
    'manufacturer_description':          'manufacturer_description',
    'names.default.value':               'default_name',
    'names.livhaven.value':              'livhaven_name',
    'shortdescriptions.default.value':   'default_short_description',
    'shortdescriptions.livhaven.value':  'livhaven_short_description',
    'descriptions.mro.value':            'mro_description',
    'descriptions.livhaven.value':       'livhaven_description',
}

DTYPE_MAP = {
    'supplier_catalog_key':       'Float64',
    'supplier_name':              'string',
    'sku':                        'string',
    'attribute_family':           'string',
    'category_name':              'string',
    'category_id':                'Float64',
    'brand_name':                 'string',
    'manufacturer_part_number':   'string',
    'last_sold_price':            'Float64',
    'item_weight':                'Float64',
    'attribute_table':            'string',
    'downloads':                  'string',
    'manufacturer_description':   'string',
    'default_name':               'string',
    'livhaven_name':              'string',
    'default_short_description':  'string',
    'livhaven_short_description': 'string',
    'mro_description':            'string',
    'livhaven_description':       'string',
}

data = pd.read_csv(RAW_PATH, dtype=str)
data.columns = data.columns.str.lower().str.strip()
data = data.rename(columns=COLUMN_MAP)

# Keep only the renamed columns — everything else is dropped
keep_cols = [c for c in COLUMN_MAP.values() if c in data.columns]
data = data[keep_cols]

valid_dtype_map = {k: v for k, v in DTYPE_MAP.items() if k in data.columns}
data = data.astype(valid_dtype_map)

print(f'Loaded. Shape: {data.shape}')
print(f'Columns: {list(data.columns)}')

---
## 2. Normalize `brand_name`

Parker appears as 5+ variants in the raw data (Parker, Parker Pneumatic Division,
Parker FRL, Parker Finite, Parker Hose). All refer to the same manufacturer.

We store the original in `brand_name_raw` for audit purposes.
Expand `BRAND_ALIASES` as new variants are discovered on the full dataset.

In [ ]:
BRAND_ALIASES = {
    'parker pneumatic division':   'Parker',
    'parker pneumatic':            'Parker',
    'parker frl':                  'Parker',
    'parker finite':               'Parker',
    'parker hose':                 'Parker',
    'parker hannifin':             'Parker',
    'parker-hannifin':             'Parker',
    'parker':                      'Parker',
    'bosch rexroth':               'Bosch Rexroth',
    'rexroth':                     'Bosch Rexroth',
    'smc corporation':             'SMC',
    'smc corp':                    'SMC',
    'smc':                         'SMC',
    'versa products':              'Versa Products',
    'versa products co.':          'Versa Products',
    'aventics':                    'Aventics',
    'hydac':                       'Hydac',
    'balluff, inc.':               'Balluff',
    'balluff':                     'Balluff',
    'hengst filtration':           'Hengst Filtration',
    'schroeder industries':        'Schroeder Industries',
    'bijur delimon international': 'Bijur Delimon',
    'bijur delimon':               'Bijur Delimon',
}

data['brand_name_raw'] = data['brand_name'].copy()

def normalize_brand(name):
    if pd.isna(name) or str(name).strip() == '':
        return pd.NA
    key = str(name).strip().lower()
    return BRAND_ALIASES.get(key, str(name).strip())

data['brand_name'] = data['brand_name'].apply(normalize_brand).astype('string')

changed = data[
    data['brand_name'] != data['brand_name_raw']
][['brand_name_raw', 'brand_name']].drop_duplicates()

print(f'Brands normalized: {len(changed)}')
print(changed.to_string())
print(f'\nUnique brands after normalization: {data["brand_name"].nunique()}')

---
## 3. Normalize `manufacturer_part_number`

Confirmed in sample: no spaces or lowercase in the raw data.
We still uppercase and strip as a safety measure on the full dataset.
Outliers (length < 4 or > 30) are flagged for human review, not dropped.

In [ ]:
data['manufacturer_part_number'] = (
    data['manufacturer_part_number']
    .astype('string')
    .str.strip()
    .str.upper()
)

pn_len = data['manufacturer_part_number'].str.len()
data['pn_flag'] = pd.NA
data.loc[pn_len < 4,  'pn_flag'] = 'too_short'
data.loc[pn_len > 30, 'pn_flag'] = 'too_long'

flagged = data['pn_flag'].notna().sum()
print(f'Part numbers flagged: {flagged}')
if flagged > 0:
    print(data[data['pn_flag'].notna()][
        ['manufacturer_part_number', 'brand_name', 'pn_flag']
    ].head(10).to_string())

---
## 4. Numeric Sanity Checks

`last_sold_price`: zero prices are bad data, replaced with NaN.  
`item_weight`: zeros are flagged but not auto-nulled.

In [ ]:
for col in ['last_sold_price', 'item_weight']:
    s = pd.to_numeric(data[col], errors='coerce')
    data[col] = s
    print(f'{col}:')
    print(f'  null:     {s.isna().sum():,} ({s.isna().mean()*100:.1f}%)')
    print(f'  zero:     {(s==0).sum():,}')
    print(f'  negative: {(s<0).sum():,}')
    print(f'  median:   {s.median():.2f}')
    print(f'  p99:      {s.quantile(0.99):.2f}')
    print()

data['last_sold_price'] = data['last_sold_price'].replace(0, np.nan)
print('Zero prices set to NaN.')

---
## 5. Parse `attribute_table`

Format confirmed by inspection: pipe-delimited key/value pairs.
`Label | | | | Value | | | | Label | | | | Value ...`

We keep only keys present in >= 1% of rows to avoid a very sparse column explosion.
All extracted columns are prefixed `attr_`.
The raw `attribute_table` column is dropped after extraction.

In [ ]:
def parse_pipe_attrs(raw):
    if pd.isna(raw) or str(raw).strip() == '':
        return {}
    parts = [p.strip() for p in str(raw).split('|')]
    parts = [p for p in parts if p]
    result = {}
    i = 0
    while i < len(parts) - 1:
        key = parts[i].strip()
        val = parts[i + 1].strip()
        if key:
            result[key] = val if val else np.nan
        i += 2
    return result

print('Parsing attribute_table (~60s on full dataset)...')
parsed    = data['attribute_table'].apply(parse_pipe_attrs)
attr_df   = pd.DataFrame(parsed.tolist(), index=data.index)
coverage  = attr_df.notna().sum().sort_values(ascending=False)
MIN_ROWS  = max(1, int(len(data) * 0.01))
keep_keys = coverage[coverage >= MIN_ROWS].index.tolist()

data = pd.concat([data, attr_df[keep_keys].add_prefix('attr_')], axis=1)
data.drop(columns=['attribute_table'], inplace=True)

print(f'Attribute columns kept (>= {MIN_ROWS} rows): {len(keep_keys)}')
print('Top 10:')
print(coverage.head(10).to_string())
print(f'\nShape: {data.shape}')

---
## 6. Final State & Export

In [ ]:
print(f'Final shape: {data.shape}')
print()
print(f'{"Column":<45} {"Filled":>8}  {"% Filled":>9}')
print('-' * 65)
for c in data.columns:
    filled = data[c].notna().sum()
    pct    = filled / len(data) * 100
    print(f'{c:<45} {filled:>8,}  {pct:>8.1f}%')

data.to_csv(CLEAN_PATH, index=False)
print(f'\nSaved → {CLEAN_PATH}')